# Классификация фотографий кошек и собак

## 1. Суть, цель и задачи проекта

**Суть:** свёрточная нейронная сеть анализирует фотографию и определяет, изображена на ней кошка или собака.

**Цель:** разработать и обучить собственный классификатор с нуля, оценить его на новых изображениях и продемонстрировать распознавание своей фотографии.

**Задачи:** проверить и очистить данные; разделить их на обучение, валидацию и тест;
подготовить изображения; построить CNN; обучить её; изучить качество и ошибки.

Датасет содержит около 25 000 фотографий. Проект объединяет весь эксперимент и демонстрацию в одном ноутбуке.


## 2. Техническое описание

Используются TensorFlow/Keras и собственная CNN из четырёх свёрточных блоков.
Веса инициализируются случайно; внешние модели и предобученные веса не используются.
Метки определяются по именам: `cat.*` — 0, `dog.*` — 1. Реальные расширения проверяются при чтении данных.
После удаления точных дубликатов данные делятся на train / validation / test в пропорции 70 / 15 / 15.
RGB-изображения уменьшаются до 128×128; нормализация /255 выполняется внутри сети.
Сеть выдаёт одну оценку `p_dog`; `p_cat = 1 − p_dog`. Порог классификации фиксирован: больше 0,5 — собака.

### 2.1. Среда и воспроизводимость

Импорты подключают обработку изображений, построение сети, метрики и графики.
Seed 42 фиксирует разделение и начальную случайность обучения. Разные GPU и версии библиотек могут дать небольшие различия.
GPU ускоряет свёртки; ячейка показывает доступное устройство.


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import io
import re
import json
import random
import hashlib
import zipfile
import shutil
import warnings
from pathlib import Path, PurePosixPath
from tempfile import mkdtemp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay)
from IPython.display import display, Markdown
from google.colab import drive, output, files

# Настройки публичной версии. Данные и веса не входят в репозиторий.
PROJECT_DIR = Path('/content/drive/MyDrive/cats_dogs_cnn')
DRIVE_ZIP_PATH = PROJECT_DIR / 'data' / 'train.zip'
SAVED_RUN_DIR = PROJECT_DIR / 'runs' / 'default'
FORCE_RETRAIN = False
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
CLASS_NAMES = ['Кошка', 'Собака']
CLASS_TO_INDEX = {'cat': 0, 'dog': 1}
WORK_DIR = Path('/content/cats_dogs_project')
WORK_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR = Path(mkdtemp(prefix='results_', dir=WORK_DIR))

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU:', [gpu.name for gpu in gpus] if gpus else 'нет — используется CPU')
print('Классы:', CLASS_TO_INDEX, '; выход sigmoid = оценка класса dog (1)')

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
fig, ax = plt.subplots(figsize=(14, 2.5))
stages = ['ZIP на Drive', 'Проверка\nи очистка', 'Разделение\n70 / 15 / 15',
          'RGB 128×128\nи пакеты', 'Обучение\nсвоей CNN', 'Тест\nи фотография']
for i, title in enumerate(stages):
    ax.text(i, 0.5, title, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.7', fc='#e8f0fc', ec='#548bd4'))
    if i < len(stages) - 1:
        ax.annotate('', xy=(i + 0.72, 0.5), xytext=(i + 0.3, 0.5),
                    arrowprops=dict(arrowstyle='->', color='#548bd4', lw=2))
ax.set(xlim=(-0.5, 5.7), ylim=(0, 1), title='Последовательность эксперимента')
ax.axis('off')
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'process_overview.png', bbox_inches='tight')
plt.show()


### 2.2. Общая обработка изображения и безопасная распаковка
Одни и те же функции используются для датасета и фотографии пользователя:
учитываем ориентацию EXIF, переводим в RGB, изменяем размер до 128×128 билинейным методом.
Изменение размера растягивает изображение; это простое учебное решение.
Здесь значения остаются в диапазоне 0–255. **Нормализация /255 находится внутри модели**,
поэтому при распознавании не нужно делить на 255 ещё раз.
Распаковка поддерживает вложенные папки и пропускает служебные файлы macOS.


In [ ]:
def read_rgb(source):
    """source: путь или BytesIO. Одинаковый способ декодирования во всём проекте."""
    with Image.open(source) as image:
        if getattr(image, 'n_frames', 1) != 1:
            raise ValueError('Ожидается одно изображение, а не анимация')
        return ImageOps.exif_transpose(image).convert('RGB').copy()

def preprocess_image(source):
    rgb = read_rgb(source)
    resized = rgb.resize(IMG_SIZE, Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32)

def safe_extract_zip(archive_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    extracted = 0
    with zipfile.ZipFile(archive_path) as archive:
        # Проверяем все пути до записи первого файла.
        members = []
        for member in archive.infolist():
            normalized = member.filename.replace('\\', '/')
            parts = PurePosixPath(normalized).parts
            if ('__MACOSX' in parts or any(p.startswith('._') for p in parts)
                    or '.DS_Store' in parts):
                continue
            if (PurePosixPath(normalized).is_absolute() or '..' in parts
                    or any(':' in p for p in parts)):
                raise ValueError('Недопустимый путь внутри ZIP: ' + member.filename)
            target = (destination / normalized).resolve()
            if not target.is_relative_to(destination):
                raise ValueError('Путь выходит за папку распаковки')
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Символические ссылки в ZIP не поддерживаются')
            members.append((member, target))
        for member, target in members:
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open('wb') as sink:
                    shutil.copyfileobj(source, sink)
                extracted += 1
    return extracted

def rounded_percentages(p_dog):
    # Округляем собаку, затем берём дополнение: сумма всегда 100.0%.
    p_dog = float(p_dog)
    if not np.isfinite(p_dog) or not 0 <= p_dog <= 1:
        raise ValueError('Некорректный выход модели')
    dog_tenths = int(np.floor(p_dog * 1000 + 0.5))
    return (1000 - dog_tenths) / 10, dog_tenths / 10

def check_model_contract(candidate):
    # Проверка договорённостей нашей сети; веса загружаются только из своего проекта.
    if tuple(candidate.input_shape[1:]) != (128, 128, 3):
        raise ValueError('Модель должна принимать RGB 128×128')
    if tuple(candidate.output_shape[1:]) != (1,):
        raise ValueError('Ожидается один выход — оценка dog')
    normalization = candidate.get_layer('normalize_0_1')
    if not isinstance(normalization, keras.layers.Rescaling):
        raise ValueError('Отсутствует нормализация внутри модели')
    if not np.isclose(normalization.scale, 1 / 255) or normalization.offset != 0:
        raise ValueError('Несовместимая нормализация')
    head = candidate.get_layer('dog_probability')
    if candidate.layers[-1] is not head or head.get_config()['activation'] != 'sigmoid':
        raise ValueError('Выходной слой должен называться dog_probability и использовать sigmoid')


## 3. Получение данных
### 3.1. Архив с Google Drive или загрузка с компьютера

В следующей ячейке выберите `drive` или `upload`. По умолчанию ZIP находится в
`Мой диск/cats_dogs_cnn/data/train.zip`; путь задаётся в разделе 2.1.
Drive также хранит checkpoint и историю между сеансами. Данные не скачиваются из репозитория.
SHA-256 вычисляется для вашего архива и затем участвует в проверке кэша модели.
Архив копируется на локальный диск Colab, чтобы обучение не читало тысячи файлов через Drive.


In [ ]:
drive.mount('/content/drive')
DATA_SOURCE = 'drive'  # 'drive' или 'upload'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if DATA_SOURCE == 'upload':
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Загрузите ровно один ZIP-архив')
    source_archive = Path(next(iter(uploaded)))
    del uploaded  # не держим копию архива в памяти
elif DATA_SOURCE == 'drive':
    source_archive = DRIVE_ZIP_PATH
else:
    raise ValueError("DATA_SOURCE: выберите 'drive' или 'upload'")
if not source_archive.is_file() or not zipfile.is_zipfile(source_archive):
    raise ValueError('Не найден ZIP-архив: ' + str(source_archive))
DATASET_SHA256 = sha256_file(source_archive)
archive_path = WORK_DIR / f'dataset_{DATASET_SHA256[:16]}.zip'
if not archive_path.is_file() or sha256_file(archive_path) != DATASET_SHA256:
    shutil.copyfile(source_archive, archive_path)
if sha256_file(archive_path) != DATASET_SHA256:
    raise ValueError('Контрольная сумма локальной копии не совпала')
print('SHA-256:', DATASET_SHA256)
print(f'Размер: {archive_path.stat().st_size / 1024**2:.1f} МиБ')


### 3.2. Распаковка
Каждый запуск распаковывает архив в новую папку: старые изображения не смешиваются с новым датасетом.
Проверка CRC выполняется самим чтением ZIP; ошибка повреждённого архива остановит ячейку.


In [ ]:
DATA_DIR = Path(mkdtemp(prefix='dataset_', dir=WORK_DIR))
file_count = safe_extract_zip(archive_path, DATA_DIR)
print('Распаковано файлов:', file_count)
print('Папка:', DATA_DIR)


## 4. Обзор и очистка данных
### 4.1. Имена, расширения и баланс классов
Ищем фотографии рекурсивно, чтобы поддержать вложенность `train/train`.
Поддерживаются `.jpg`, `.jpeg`, `.png`; метка читается только из начала имени `cat.` или `dog.`.
Служебные файлы не участвуют. Если класс определяется, но расширение другое,
покажем его отдельно. Отсутствие одного из классов останавливает подготовку.


In [ ]:
def discover_images(data_dir):
    rows, ignored = [], []
    for path in sorted(Path(data_dir).rglob('*')):
        if not path.is_file():
            continue
        if '__MACOSX' in path.parts or path.name.startswith('._') or path.name == '.DS_Store':
            continue
        match = re.match(r'^(cat|dog)\.', path.name, flags=re.IGNORECASE)
        if not match or path.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
            ignored.append(str(path))
            continue
        prefix = match.group(1).lower()
        rows.append({'path': str(path), 'filename': path.name,
                     'extension': path.suffix.lower(), 'label': CLASS_TO_INDEX[prefix]})
    frame = pd.DataFrame(rows, columns=['path', 'filename', 'extension', 'label'])
    if frame.empty or set(frame['label']) != {0, 1}:
        raise ValueError('Не найдены оба класса cat.* и dog.* в JPG/PNG')
    return frame, ignored

raw_df, ignored_files = discover_images(DATA_DIR)
print('Всего подходящих имён:', len(raw_df))
print('Расширения:', raw_df['extension'].value_counts().to_dict())
print('Кошек:', int((raw_df['label'] == 0).sum()))
print('Собак:', int((raw_df['label'] == 1).sum()))
print('Прочих файлов:', len(ignored_files))
if ignored_files:
    print('Примеры пропущенных файлов:', ignored_files[:5])
raw_df.to_csv(ARTIFACT_DIR / 'raw_inventory.csv', index=False)


### 4.2. Повреждённые изображения и точные дубликаты
Проверяем структуру файла и полностью декодируем каждый снимок. Неисправные изображения исключаем,
а их имена и причины сохраняем. SHA-256 файла находит побайтовые копии;
SHA-256 **размеров и исходных RGB-пикселей до уменьшения** также находит одинаковые изображения,
которые отличаются только метаданными. Все файлы читаются по одному: 25 000 снимков не держим в RAM.

На каждый одинаковый снимок оставляем один экземпляр **до разделения**. Если одинаковым пикселям
присвоены разные классы, исключаем всю противоречивую группу. После этого один и тот же снимок
не может попасть в разные выборки. Почти одинаковые кадры, JPEG с другим сжатием и серии снимков
этот метод не распознаёт — это ограничение проекта. Проверка может занять несколько минут.


In [ ]:
def audit_images(frame, verbose=True):
    valid, broken = [], []
    for index, row in enumerate(frame.to_dict('records'), start=1):
        path = Path(row['path'])
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('error', Image.DecompressionBombWarning)
                with Image.open(path) as image:
                    image_format = image.format
                    image.verify()
                rgb = read_rgb(path)  # полная загрузка, включая EXIF → RGB
                pixels_hash = hashlib.sha256()
                pixels_hash.update(f'{rgb.width}x{rgb.height}:RGB:'.encode('ascii'))
                pixels_hash.update(rgb.tobytes())
                file_hash = hashlib.sha256(path.read_bytes()).hexdigest()
            valid.append({**row, 'format': image_format,
                          'width': rgb.width, 'height': rgb.height,
                          'file_sha256': file_hash,
                          'pixel_sha256': pixels_hash.hexdigest()})
        except Exception as error:
            broken.append({**row, 'reason': f'{type(error).__name__}: {error}'})
        if verbose and index % 2500 == 0:
            print(f'Проверено {index}/{len(frame)}')
    valid_df = pd.DataFrame(valid)
    broken_df = pd.DataFrame(broken, columns=list(frame.columns) + ['reason'])
    if valid_df.empty:
        raise ValueError('Нет пригодных изображений')
    return valid_df, broken_df

def remove_duplicates(valid_df):
    label_counts = valid_df.groupby('pixel_sha256')['label'].nunique()
    conflict_hashes = set(label_counts[label_counts > 1].index)
    conflicts = valid_df[valid_df['pixel_sha256'].isin(conflict_hashes)].copy()
    nonconflicting = valid_df[~valid_df['pixel_sha256'].isin(conflict_hashes)].copy()
    duplicates = nonconflicting[nonconflicting.duplicated('pixel_sha256', keep='first')].copy()
    clean = nonconflicting.drop_duplicates('pixel_sha256', keep='first').reset_index(drop=True)
    if set(clean['label']) != {0, 1}:
        raise ValueError('После очистки нужны оба класса')
    return clean, duplicates, conflicts

valid_df, broken_df = audit_images(raw_df)
clean_df, duplicate_df, conflict_df = remove_duplicates(valid_df)
byte_duplicates = int(valid_df.duplicated('file_sha256').sum())
print('Реальные форматы внутри файлов:', valid_df['format'].value_counts().to_dict())
print('Повреждённых:', len(broken_df))
print('Лишних побайтовых копий:', byte_duplicates)
print('Исключено повторных RGB-изображений:', len(duplicate_df))
print('Исключено файлов с противоречивыми метками:', len(conflict_df))
print('Осталось уникальных:', len(clean_df))
for filename, frame in [('valid_inventory.csv', valid_df), ('broken_files.csv', broken_df),
                        ('duplicates.csv', duplicate_df), ('label_conflicts.csv', conflict_df),
                        ('clean_inventory.csv', clean_df)]:
    frame.to_csv(ARTIFACT_DIR / filename, index=False)


### 4.3. Визуальный обзор данных

Примеры показывают разнообразие поз, фона и освещения и помогают проверить метки.
Баланс классов позволяет увидеть, не преобладает ли один ответ. Гистограммы размеров объясняют необходимость единого формата входа.
Диаграмма очистки показывает, сколько изображений остаётся после проверок; исключённые категории не пересекаются.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for label in [0, 1]:
    examples = clean_df[clean_df['label'] == label].sample(
        n=min(4, int((clean_df['label'] == label).sum())), random_state=SEED)
    for ax in axes[label]:
        ax.axis('off')
    for ax, row in zip(axes[label], examples.to_dict('records')):
        ax.imshow(read_rgb(row['path']))
        ax.set_title(f"{CLASS_NAMES[label]}: {row['filename']}", fontsize=9)
plt.tight_layout()
fig.savefig(ARTIFACT_DIR / 'data_examples.png', dpi=160, bbox_inches='tight')
plt.show()

counts = clean_df['label'].value_counts().reindex([0, 1], fill_value=0)
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(CLASS_NAMES, counts.values, color=['#548bd4', '#e6a34a'])
ax.set_ylabel('Количество уникальных изображений')
ax.set_title('Баланс классов после очистки')
for i, value in enumerate(counts.values):
    ax.text(i, value, str(value), ha='center', va='bottom')
ax.margins(y=0.15)
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'class_balance.png', dpi=160)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(valid_df['width'], bins=35, alpha=0.7, label='Ширина')
axes[0].hist(valid_df['height'], bins=35, alpha=0.7, label='Высота')
axes[0].set(xlabel='Пиксели', ylabel='Число фотографий', title='Исходные размеры')
axes[0].legend()
axes[1].hist(valid_df['width'] / valid_df['height'], bins=35, color='#548bd4')
axes[1].set(xlabel='Ширина / высота', ylabel='Число фотографий', title='Пропорции изображений')
categories = ['Повреждения', 'Конфликты', 'Лишние копии', 'Осталось']
values = [len(broken_df), len(conflict_df), len(duplicate_df), len(clean_df)]
axes[2].barh(categories, values, color=['#dd6666', '#dd9966', '#aaaaaa', '#548bd4'])
for i, value in enumerate(values):
    axes[2].text(value, i, f' {value}', va='center')
axes[2].set(xlabel='Количество', title='Результат очистки')
axes[2].margins(x=0.18)
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'data_audit.png', bbox_inches='tight')
plt.show()


## 5. Подготовка к обучению
### 5.1. Стратифицированное разделение 70/15/15
Отдельно перемешиваем фотографии каждого класса с фиксированным seed.
От каждого класса примерно 70% направляем в train, 15% в validation и остальные в test.
Небольшое отклонение от процентов возможно из-за округления до целого числа изображений.
Так сохраняется соотношение кошек и собак. Проверяем непересечение хешей всех трёх наборов.
Train обновляет веса, validation выбирает лучшую эпоху, test оценивает уже выбранную модель.


In [ ]:
def stratified_split(frame, seed=SEED):
    rng = np.random.default_rng(seed)
    pieces = {'train': [], 'validation': [], 'test': []}
    for label in [0, 1]:
        group = frame[frame['label'] == label]
        n = len(group)
        if n < 10:
            raise ValueError('Для разделения нужно хотя бы 10 уникальных фото каждого класса')
        order = rng.permutation(n)
        n_train, n_val = round(n * 0.70), round(n * 0.15)
        for name, indices in [('train', order[:n_train]),
                              ('validation', order[n_train:n_train + n_val]),
                              ('test', order[n_train + n_val:])]:
            pieces[name].append(group.iloc[indices])
    return tuple(pd.concat(pieces[name]).sample(frac=1, random_state=seed).reset_index(drop=True)
                 for name in ['train', 'validation', 'test'])

train_df, val_df, test_df = stratified_split(clean_df)
hash_sets = [set(frame['pixel_sha256']) for frame in [train_df, val_df, test_df]]
assert all(hash_sets[i].isdisjoint(hash_sets[j]) for i, j in [(0, 1), (0, 2), (1, 2)])
assert sum(map(len, [train_df, val_df, test_df])) == len(clean_df)
assert len(set.union(*hash_sets)) == len(clean_df)
split_rows = []
for name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    frame.to_csv(ARTIFACT_DIR / f'{name}_split.csv', index=False)
    split_rows.append({'Набор': name, 'Всего': len(frame),
                       'Кошки': int((frame['label'] == 0).sum()),
                       'Собаки': int((frame['label'] == 1).sum()),
                       'Доля, %': round(100 * len(frame) / len(clean_df), 2)})
split_summary = pd.DataFrame(split_rows)
display(split_summary)
split_summary.to_csv(ARTIFACT_DIR / 'split_summary.csv', index=False)
print('Проверка пройдена: пересечений одинаковых изображений нет.')

fig, ax = plt.subplots(figsize=(8, 4))
positions = np.arange(3)
ax.bar(positions, split_summary['Кошки'], label='Кошка', color='#548bd4')
ax.bar(positions, split_summary['Собаки'], bottom=split_summary['Кошки'],
       label='Собака', color='#e6a34a')
for i, row in enumerate(split_summary.to_dict('records')):
    ax.text(i, row['Всего'], f"{row['Всего']} ({row['Доля, %']:.1f}%)", ha='center', va='bottom')
ax.set(xticks=positions, xticklabels=['Train', 'Validation', 'Test'],
       ylabel='Изображения', title='Разделение данных с сохранением баланса классов')
ax.margins(y=0.18)
ax.legend()
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'dataset_split.png', bbox_inches='tight')
plt.show()


### 5.2. Предобработка и загрузка пакетами

`tf.data` читает фотографии по мере необходимости и собирает пакеты по 32. Перемешиваются только пути train;
validation и test сохраняют порядок. Prefetch готовит следующий пакет параллельно вычислениям.
Общая функция Pillow выполняет EXIF → RGB → 128×128, поэтому обучающие и пользовательские фотографии обрабатываются одинаково.
График сравнивает исходник, уменьшенный вход и интенсивности до и после /255. Нормализация здесь показана для объяснения; в обучение передаётся 0–255, а деление выполняет слой сети.


In [ ]:
def decode_path(path_tensor):
    path_value = path_tensor.item() if isinstance(path_tensor, np.ndarray) else path_tensor
    if isinstance(path_value, bytes):
        path_value = path_value.decode('utf-8')
    return preprocess_image(str(path_value))

def load_sample(path, label):
    image = tf.numpy_function(decode_path, [path], tf.float32)
    image.set_shape((128, 128, 3))
    return image, tf.cast(label, tf.float32)

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (frame['path'].to_numpy(dtype=str), frame['label'].to_numpy(dtype=np.int32)))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load_sample, num_parallel_calls=2, deterministic=True)
    return dataset.batch(BATCH_SIZE).prefetch(1)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)
images_batch, labels_batch = next(iter(train_ds))
print('Форма пакета:', images_batch.shape, '; метки:', labels_batch.numpy()[:8])
print('Диапазон до слоя нормализации:', float(tf.reduce_min(images_batch)),
      float(tf.reduce_max(images_batch)))
assert images_batch.shape[1:] == (128, 128, 3)
assert set(np.unique(labels_batch.numpy())).issubset({0, 1})

sample_path = train_df.iloc[0]['path']
prepared_sample = preprocess_image(sample_path)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(read_rgb(sample_path)); axes[0].set_title('Исходник с учётом EXIF')
axes[1].imshow(prepared_sample.astype(np.uint8)); axes[1].set_title('RGB 128×128')
for ax in axes[:2]:
    ax.axis('off')
axes[2].hist(prepared_sample.ravel(), bins=32, color='#548bd4')
axes[2].set(title='Вход сети: 0–255', xlabel='Интенсивность', ylabel='Число значений')
axes[3].hist((prepared_sample / 255).ravel(), bins=32, color='#e6a34a')
axes[3].set(title='После Rescaling: 0–1', xlabel='Нормированное значение')
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'preprocessing.png', bbox_inches='tight')
plt.show()


## 6. Собственная CNN
### 6.1. Архитектура и назначение слоёв
Все обучаемые веса создаются **случайно с нуля**. Аугментации (отражение по горизонтали,
небольшие повороты и масштабирование) работают внутри сети **только при training=True**.
При validation, test и predict они отключены; Dropout тоже отключён.

- Rescaling переводит 0–255 в 0–1 и сохраняется вместе с моделью.
- Conv2D ищет локальные признаки; ReLU позволяет описывать нелинейные зависимости.
- MaxPooling уменьшает размер карты признаков вдвое, снижая вычисления.
- Четыре блока постепенно увеличивают число признаков: 32 → 64 → 128 → 128.
- Flatten разворачивает карты признаков, Dense объединяет их для классификации.
- Dropout временно выключает часть признаков при обучении, уменьшая запоминание.
- Один sigmoid выдаёт `p_dog`: **0 = кошка, 1 = собака**, поэтому `p_cat = 1 − p_dog`.
  Порог фиксирован заранее: `p_dog > 0.5` → собака; иначе кошка
  (при ровно 0.5 выбираем кошку, как стандартная метрика BinaryAccuracy).

Схема: **128×128×3 → 64×64×32 → 32×32×64 → 16×16×128 → 8×8×128
→ 8192 → Dense(64) → sigmoid(1)**.


In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal', seed=SEED),
    keras.layers.RandomRotation(0.05, seed=SEED + 1),
    keras.layers.RandomZoom(0.10, seed=SEED + 2),
], name='train_only_augmentation')

model = keras.Sequential([
    keras.Input(shape=(128, 128, 3), name='rgb_0_255'),
    keras.layers.Rescaling(1.0 / 255, name='normalize_0_1'),
    augmentation,
    keras.layers.Conv2D(32, 3, padding='same', activation='relu', name='conv_1'),
    keras.layers.MaxPooling2D(name='pool_1'),
    keras.layers.Conv2D(64, 3, padding='same', activation='relu', name='conv_2'),
    keras.layers.MaxPooling2D(name='pool_2'),
    keras.layers.Conv2D(128, 3, padding='same', activation='relu', name='conv_3'),
    keras.layers.MaxPooling2D(name='pool_3'),
    keras.layers.Conv2D(128, 3, padding='same', activation='relu', name='conv_4'),
    keras.layers.MaxPooling2D(name='pool_4'),
    keras.layers.Dropout(0.25, seed=SEED + 3),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu', name='dense_features'),
    keras.layers.Dropout(0.5, seed=SEED + 4),
    keras.layers.Dense(1, activation='sigmoid', name='dog_probability'),
], name='cats_dogs_cnn_from_scratch')
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
check_model_contract(model)
model.summary()
summary_lines = []
model.summary(print_fn=lambda line, **kwargs: summary_lines.append(line))
(ARTIFACT_DIR / 'architecture.txt').write_text('\n'.join(summary_lines), encoding='utf-8')

# Сохраняем схему для презентации. Если Graphviz недоступен, summary выше остаётся рабочим.
try:
    keras.utils.plot_model(model, to_file=str(ARTIFACT_DIR / 'architecture.png'),
                           show_shapes=True, show_layer_names=True, dpi=100)
    display(Image.open(ARTIFACT_DIR / 'architecture.png'))
except (ImportError, OSError, ValueError) as error:
    print('Схема Graphviz недоступна. Используйте текстовую схему и summary:', error)


### 6.2. Проверки сети и примеры аугментаций

Проверяются форма выхода, диапазон sigmoid и детерминированность при `training=False`.
Dropout и аугментации отключены при проверке и распознавании; `/255` присутствует ровно один раз.
Демонстрация показывает случайные преобразования одного обучающего снимка. Они создают новые варианты входа,
сохраняя класс, и уменьшают зависимость сети от конкретной позы. Это проверка устройства сети, а не её качества до обучения.


In [ ]:
probe = preprocess_image(train_df.iloc[0]['path'])[None, ...]
first = model(probe, training=False).numpy()
second = model(probe, training=False).numpy()
assert first.shape == (1, 1) and np.all((first >= 0) & (first <= 1))
np.testing.assert_allclose(first, second, rtol=0, atol=1e-7)
np.testing.assert_allclose(
    model.get_layer('normalize_0_1')(probe).numpy(), probe / 255, atol=1e-7)
for probability in [0, 0.12345, 0.5, 0.9995, 1]:
    cat_percent, dog_percent = rounded_percentages(probability)
    assert round(cat_percent + dog_percent, 1) == 100.0
print('Проверки формы выхода, нормализации и процентов пройдены.')

fig, axes = plt.subplots(1, 5, figsize=(13, 3))
axes[0].imshow(probe[0].astype(np.uint8)); axes[0].set_title('Исходный вход')
demo_augmentation = keras.models.clone_model(augmentation)
for ax in axes[1:]:
    altered = demo_augmentation(probe / 255.0, training=True).numpy()[0]
    ax.imshow(np.clip(altered, 0, 1)); ax.set_title('Аугментация train')
for ax in axes:
    ax.axis('off')
fig.suptitle('Случайные отражения, небольшие повороты и масштабирование')
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'augmentation.png', bbox_inches='tight')
plt.show()
del demo_augmentation


## 7. Обучение
### 7.1. Обновление весов и выбор лучшей эпохи

Эпоха — один проход по train. Adam обновляет веса по градиентам binary crossentropy: уверенная ошибка штрафуется сильнее.
Validation не обновляет веса, а оценивает способность обобщать. ModelCheckpoint сохраняет модель с минимальным val_loss;
EarlyStopping останавливает обучение после четырёх эпох без улучшения. Предел — 20 эпох.
После обучения загружается лучший checkpoint. Test не участвует в выборе модели или параметров.


Лучший checkpoint и журнал сохраняются в папке проекта на Google Drive. Первый запуск обучает CNN с нуля; следующие полные запуски используют собственные сохранённые веса и журнал. Это не внешние предобученные веса. Test по-прежнему не участвует в выборе лучшей эпохи.

Кэш повторно используется только при совпадении архива, seed, архитектуры и настроек. Для нового обучения задайте `FORCE_RETRAIN = True` в разделе 2.1. При изменении алгоритма подготовки данных обновите `split_policy`.


In [ ]:
EPOCHS = 20
# Постоянный checkpoint своей CNN: переживает отключение и удаление среды Colab.
if not Path('/content/drive/MyDrive').exists():
    raise RuntimeError('Сначала выполните ячейку подключения Google Drive')
SAVED_RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = SAVED_RUN_DIR / 'best_model.keras'
SAVED_HISTORY_PATH = SAVED_RUN_DIR / 'history.json'
RUN_CONFIG = {
    'dataset_sha256': DATASET_SHA256, 'seed': SEED,
    'image_size': list(IMG_SIZE), 'batch_size': BATCH_SIZE,
    'epochs_limit': EPOCHS, 'learning_rate': 0.001, 'patience': 4,
    'tensorflow_version': tf.__version__,
    'architecture_sha256': hashlib.sha256(model.to_json().encode()).hexdigest(),
    'split_policy': 'rgb-dedup-stratified-70-15-15-v1',
}
RUN_CONFIG_PATH = SAVED_RUN_DIR / 'run_config.json'
def cache_is_compatible(config_path, expected):
    try:
        return json.loads(Path(config_path).read_text(encoding='utf-8')) == expected
    except (OSError, ValueError):
        return False

use_cache = (not FORCE_RETRAIN and BEST_MODEL_PATH.is_file()
             and SAVED_HISTORY_PATH.is_file()
             and cache_is_compatible(RUN_CONFIG_PATH, RUN_CONFIG))
if use_cache:
    model = keras.models.load_model(BEST_MODEL_PATH)
    history_dict = json.loads(SAVED_HISTORY_PATH.read_text(encoding='utf-8'))
    print('Загружены собственная обученная CNN и журнал с Google Drive; повторное обучение не требуется.')
else:
    # До начала нового fit снимаем старую отметку о согласованности.
    # Прерванное обучение не оставит валидный кэш от прошлого запуска.
    RUN_CONFIG_PATH.unlink(missing_ok=True)
    callbacks = [
        keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), monitor='val_loss',
                                        mode='min', save_best_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', patience=4,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.TerminateOnNaN(),
        keras.callbacks.CSVLogger(str(SAVED_RUN_DIR / 'training_log.csv')),
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
    history_dict = {key: [float(v) for v in values] for key, values in history.history.items()}
    if not all(np.isfinite(history_dict['val_loss'])):
        raise RuntimeError('Некорректная loss; итоговую оценку не проводим')
    SAVED_HISTORY_PATH.write_text(json.dumps(history_dict, indent=2), encoding='utf-8')
    model = keras.models.load_model(BEST_MODEL_PATH)
    RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')
check_model_contract(model)
(ARTIFACT_DIR / 'history.json').write_text(json.dumps(history_dict, indent=2), encoding='utf-8')
BEST_EPOCH = int(np.argmin(history_dict['val_loss']) + 1)
print('Выбрана эпоха:', BEST_EPOCH, '; validation loss:', min(history_dict['val_loss']))
print('Постоянная модель:', BEST_MODEL_PATH)


## 8. Оценка качества
### 8.1. Кривые обучения
Accuracy — доля верных ответов; loss учитывает также уверенность и штрафует уверенные ошибки.
Если train улучшается, а validation loss растёт и validation accuracy отстаёт, сеть может
переобучаться: запоминать train и хуже работать на новых снимках.
Небольшое отличие кривых нормально. Train измеряется с аугментацией и Dropout, validation без них,
поэтому validation иногда лучше train. Пунктир обозначает эпоху выбранной модели.


In [ ]:
epochs_axis = np.arange(1, len(history_dict['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in [(axes[0], 'accuracy', 'Доля правильных ответов'),
                          (axes[1], 'loss', 'Ошибка: binary crossentropy')]:
    ax.plot(epochs_axis, history_dict[metric], label='train')
    ax.plot(epochs_axis, history_dict['val_' + metric], label='validation')
    ax.axvline(BEST_EPOCH, color='gray', linestyle='--', label='лучшая эпоха')
    ax.set(xlabel='Эпоха', ylabel=metric, title=title)
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'learning_curves.png', dpi=160)
plt.show()


### 8.2. Финальный test: accuracy, precision, recall, F1
Здесь впервые оцениваем выбранную модель на test. Порог 0.5 зафиксирован заранее.
Предсказания собираем **в том же порядке**, что и метки, без перемешивания теста.
Accuracy относится ко всему набору; для каждого класса дополнительно считаем долю верных ответов
**среди реальных изображений этого класса** — она совпадает с recall.

- Precision кошек: какая доля названных кошками снимков действительно кошки.
- Recall кошек: какую долю всех реальных кошек удалось найти. Для собак смысл аналогичен.
- F1 — гармоническое среднее precision и recall; падает, если одна из этих величин мала.
- Support — число реальных снимков данного класса в test.

Метрики записываем в процентах. После просмотра test не подбираем по нему гиперпараметры:
новые варианты сравниваем по validation. Многократный выбор по test сделает оценку необъективной.


In [ ]:
test_result = model.evaluate(test_ds, return_dict=True, verbose=1)
probabilities = model.predict(test_ds, verbose=1).reshape(-1)
y_true = test_df['label'].to_numpy(dtype=np.int32)
assert len(probabilities) == len(y_true)
assert np.all(np.isfinite(probabilities)) and np.all((probabilities >= 0) & (probabilities <= 1))
y_pred = (probabilities > 0.5).astype(np.int32)
test_accuracy = float(accuracy_score(y_true, y_pred))
np.testing.assert_allclose(test_accuracy, test_result['accuracy'], atol=1e-6)
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=[0, 1], zero_division=0)
metrics_df = pd.DataFrame({
    'Класс': CLASS_NAMES,
    'Доля верных среди реальных фото класса, % (= recall)': recall * 100,
    'Precision, %': precision * 100, 'Recall, %': recall * 100,
    'F1, %': f1 * 100, 'Support': support,
})
print(f'Test accuracy: {test_accuracy:.2%}; test loss: {test_result["loss"]:.4f}')
display(metrics_df.round(2))
metrics_df.to_csv(ARTIFACT_DIR / 'test_metrics.csv', index=False)
test_predictions = test_df.copy()
test_predictions['p_dog'] = probabilities
test_predictions['predicted_label'] = y_pred
test_predictions['correct'] = y_true == y_pred
test_predictions.to_csv(ARTIFACT_DIR / 'test_predictions.csv', index=False)
test_metrics = {'accuracy': test_accuracy, 'loss': float(test_result['loss']),
                'classes': {CLASS_NAMES[i]: {'precision': float(precision[i]),
                  'recall': float(recall[i]), 'f1': float(f1[i]), 'support': int(support[i])}
                  for i in [0, 1]}}
(ARTIFACT_DIR / 'test_metrics.json').write_text(
    json.dumps(test_metrics, ensure_ascii=False, indent=2), encoding='utf-8')


### 8.3. Confusion matrix и примеры ошибок
Строки матрицы — **реальные классы**, столбцы — предсказанные.
Диагональ показывает верные ответы, остальные клетки — перепутанные кошки и собаки.
Покажем до восьми случайных ошибок с истинной и предсказанной меткой.
Это помогает обсудить сложный ракурс, фон, освещение или частично видимое животное;
сам график не доказывает конкретную причину ошибки.


In [ ]:
matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set(xlabel='Предсказанный класс', ylabel='Реальный класс', title='Матрица ошибок: test')
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'confusion_matrix.png', dpi=160)
plt.show()
pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    ARTIFACT_DIR / 'confusion_matrix.csv')

mistakes = np.flatnonzero(y_true != y_pred)
if len(mistakes) == 0:
    print('На этом тестовом наборе ошибок не найдено.')
else:
    chosen = np.random.default_rng(SEED).choice(mistakes, min(8, len(mistakes)), replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(13, 7))
    for ax in axes.flat:
        ax.axis('off')
    for ax, index in zip(axes.flat, chosen):
        ax.imshow(read_rgb(test_df.iloc[index]['path']))
        cat_pct, dog_pct = rounded_percentages(probabilities[index])
        ax.set_title(f'Истина: {CLASS_NAMES[y_true[index]]}\n'
                     f'Ответ: {CLASS_NAMES[y_pred[index]]}\n'
                     f'Кошка {cat_pct:.1f}%; собака {dog_pct:.1f}%', fontsize=9)
    fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / 'mistakes.png', dpi=160, bbox_inches='tight')
    plt.show()


## 9. Распознавание своей фотографии
### 9.1. Предсказание

Фотография проходит ту же предобработку, что и датасет. Лучшая обученная модель работает с `training=False`.
Единственный sigmoid соответствует собаке; для кошки берётся дополнение. Округление до десятых процента сохраняет сумму 100%.
Проценты являются оценками модели для двух классов, а не гарантией правильности; для других изображений ответ может быть некорректным.


In [ ]:
def predict_photo(image_bytes, active_model):
    check_model_contract(active_model)
    prepared = preprocess_image(io.BytesIO(image_bytes))[None, ...]
    prediction = active_model(prepared, training=False).numpy()
    p_dog = float(prediction[0, 0])
    cat_pct, dog_pct = rounded_percentages(p_dog)
    original = read_rgb(io.BytesIO(image_bytes))
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(original)
    ax.axis('off')
    ax.set_title(f'Кошка — {cat_pct:.1f}%.\nСобака — {dog_pct:.1f}%.')
    fig.tight_layout()
    plt.show()
    print(f'Кошка — {cat_pct:.1f}%.')
    print(f'Собака — {dog_pct:.1f}%.')
    print('Это оценки модели, обученной различать кошек и собак; '
          'для других изображений результат может быть некорректным.')
    return p_dog


### 9.2. Демонстрация на загруженном изображении

Обычная HTML-кнопка открывает выбор JPG/PNG. Браузер читает фотографию и передаёт её Python-функции через callback Colab. Функция использует ту же предобработку и обученную модель, что и тестирование. Выводятся исходное изображение и оценки обоих классов; веса не изменяются. Кнопка не зависит от ipywidgets.

Для демонстрации после перезапуска достаточно выполнить только следующую ячейку: она содержит нужные импорты и те же функции обработки, подключает Google Drive и загружает собственную обученную CNN. Повторное обучение не требуется. При запросе Google подтвердите подключение своего Drive.


In [ ]:
import io
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
from tensorflow import keras
IMG_SIZE = (128, 128)

def read_rgb(source):
    """source: путь или BytesIO. Одинаковый способ декодирования во всём проекте."""
    with Image.open(source) as image:
        if getattr(image, 'n_frames', 1) != 1:
            raise ValueError('Ожидается одно изображение, а не анимация')
        return ImageOps.exif_transpose(image).convert('RGB').copy()

def preprocess_image(source):
    rgb = read_rgb(source)
    resized = rgb.resize(IMG_SIZE, Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32)

def rounded_percentages(p_dog):
    # Округляем собаку, затем берём дополнение: сумма всегда 100.0%.
    p_dog = float(p_dog)
    if not np.isfinite(p_dog) or not 0 <= p_dog <= 1:
        raise ValueError('Некорректный выход модели')
    dog_tenths = int(np.floor(p_dog * 1000 + 0.5))
    return (1000 - dog_tenths) / 10, dog_tenths / 10

def check_model_contract(candidate):
    # Проверка договорённостей нашей сети; веса загружаются только из своего проекта.
    if tuple(candidate.input_shape[1:]) != (128, 128, 3):
        raise ValueError('Модель должна принимать RGB 128×128')
    if tuple(candidate.output_shape[1:]) != (1,):
        raise ValueError('Ожидается один выход — оценка dog')
    normalization = candidate.get_layer('normalize_0_1')
    if not isinstance(normalization, keras.layers.Rescaling):
        raise ValueError('Отсутствует нормализация внутри модели')
    if not np.isclose(normalization.scale, 1 / 255) or normalization.offset != 0:
        raise ValueError('Несовместимая нормализация')
    head = candidate.get_layer('dog_probability')
    if candidate.layers[-1] is not head or head.get_config()['activation'] != 'sigmoid':
        raise ValueError('Выходной слой должен называться dog_probability и использовать sigmoid')

import base64
from IPython.display import display, HTML, JSON
from google.colab import output, drive

# Google Drive сохраняет обученные веса между сеансами Colab.
# Автономный режим после перезапуска. Если меняли путь сохранения,
# укажите здесь свою папку. При полном запуске берём SAVED_RUN_DIR из 2.1.
demo_run_dir = globals().get('SAVED_RUN_DIR',
    Path('/content/drive/MyDrive/cats_dogs_cnn/runs/default'))
saved_model = Path(demo_run_dir) / 'best_model.keras'
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
if saved_model.is_file():
    model = keras.models.load_model(saved_model, compile=False)
    check_model_contract(model)
    print('Обученная модель загружена с Google Drive. Можно выбрать фотографию.')
else:
    model = None
    print('Сохранённая модель пока отсутствует. Сначала завершите полный запуск.')

def classify_uploaded_photo(filename, encoded):
    try:
        if Path(filename).suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
            raise ValueError('Выберите JPG/JPEG/PNG')
        if model is None:
            raise ValueError('Модель ещё не сохранена на Drive. Дождитесь завершения обучения, затем выполните эту ячейку заново.')
        check_model_contract(model)
        image_bytes = base64.b64decode(encoded, validate=True)
        prepared = preprocess_image(io.BytesIO(image_bytes))[None, ...]
        p_dog = float(model(prepared, training=False).numpy()[0, 0])
        cat_pct, dog_pct = rounded_percentages(p_dog)
        return JSON({'message': f'Кошка — {cat_pct:.1f}%.\nСобака — {dog_pct:.1f}%.',
                     'ok': True})
    except Exception as error:
        return JSON({'message': str(error), 'ok': False})

output.register_callback('cats_dogs.classify_photo', classify_uploaded_photo)
display(HTML(r'''
<div style="font-family:Arial;max-width:650px">
  <label style="display:inline-block;background:#2879ba;color:white;padding:12px 20px;border-radius:6px;cursor:pointer">
    Загрузить JPG/PNG
    <input id="pets-photo-input" type="file" accept="image/jpeg,image/png" style="display:none">
  </label>
  <p id="pets-photo-status" style="white-space:pre-line"></p>
  <img id="pets-photo-preview" style="display:none;max-width:100%;max-height:420px" alt="Загруженная фотография">
  <p>Это оценки модели, обученной различать кошек и собак; для других изображений результат может быть некорректным.</p>
</div>
<script>
document.getElementById('pets-photo-input').addEventListener('change', async function() {
  const file = this.files[0];
  if (!file) return;
  const status = document.getElementById('pets-photo-status');
  const preview = document.getElementById('pets-photo-preview');
  status.textContent = 'Обработка фотографии…';
  this.disabled = true;
  try {
    const data = await new Promise((resolve, reject) => {
      const reader = new FileReader();
      reader.onload = () => resolve(reader.result);
      reader.onerror = () => reject(new Error('Не удалось прочитать файл'));
      reader.readAsDataURL(file);
    });
    preview.src = data;
    preview.style.display = 'block';
    const response = await google.colab.kernel.invokeFunction(
      'cats_dogs.classify_photo', [file.name, data.split(',')[1]], {});
    status.textContent = response.data['application/json'].message;
  } catch (error) {
    status.textContent = 'Не удалось распознать фотографию. Проверьте подключение к сеансу и выполните эту ячейку повторно. ' + error.message;
  } finally {
    this.disabled = false;
    this.value = '';
  }
});
</script>
'''))


## 10. Как работает обученная сеть
### 10.1. Карты признаков

Свёрточные фильтры вычисляют пространственные карты активаций. Для одного снимка train показываются восемь наиболее активных каналов
первого и последнего свёрточных блоков. Ранние слои сохраняют больше деталей, поздние объединяют их на меньшей сетке.
Свой масштаб цвета у каждой карты позволяет видеть структуру; яркость разных карт напрямую не сравнивается.
Карты не являются доказательством того, что сеть смотрит только на животное: фильтры могут реагировать на фон.


In [ ]:
feature_probe = preprocess_image(train_df.iloc[0]['path'])[None, ...]
feature_model = keras.Model(inputs=model.inputs,
                            outputs=[model.get_layer('conv_1').output,
                                     model.get_layer('conv_4').output])
feature_maps = feature_model(feature_probe, training=False)
fig, axes = plt.subplots(2, 8, figsize=(15, 5))
for row, (tensor, title) in enumerate(zip(feature_maps, ['Conv 1', 'Conv 4'])):
    maps = tensor.numpy()[0]
    channel_ids = np.argsort(maps.mean(axis=(0, 1)))[-8:][::-1]
    for ax, channel in zip(axes[row], channel_ids):
        ax.imshow(maps[:, :, channel], cmap='viridis')
        ax.set_title(f'{title}, канал {channel}', fontsize=8)
        ax.axis('off')
fig.suptitle('Активации обученных фильтров для одного снимка train')
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'feature_maps.png', bbox_inches='tight')
plt.show()


### 10.2. Уверенность и качество по классам

Распределения `p_dog` показывают, насколько отделяются реальные кошки и собаки. Перекрытие около 0,5 соответствует неоднозначным ответам.
Диаграмма метрик сравнивает precision, recall и F1 по классам. Отдельный график показывает уверенность правильных и ошибочных ответов:
даже высокий процент иногда сопровождает ошибку. Эти графики описывают финальный test и не используются для подбора порога.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for label, color in enumerate(['#548bd4', '#e6a34a']):
    axes[0].hist(probabilities[y_true == label], bins=np.linspace(0, 1, 26),
                 alpha=0.6, label=CLASS_NAMES[label], color=color)
axes[0].axvline(0.5, color='gray', ls='--')
axes[0].set(xlabel='Оценка класса «собака»', ylabel='Число фотографий', title='Разделение ответов на test')
axes[0].legend()
positions = np.arange(2)
for offset, values, title in [(-0.25, precision, 'Precision'), (0, recall, 'Recall'), (0.25, f1, 'F1')]:
    bars = axes[1].bar(positions + offset, values * 100, width=0.25, label=title)
    axes[1].bar_label(bars, fmt='%.1f', fontsize=8)
axes[1].set(xticks=positions, xticklabels=CLASS_NAMES, ylim=(0, 110), ylabel='%', title='Метрики по классам')
axes[1].legend(loc='lower center')
confidence = np.maximum(probabilities, 1 - probabilities)
correct_mask = y_pred == y_true
for mask, title, color in [(correct_mask, 'Правильно', '#548bd4'), (~correct_mask, 'Ошибка', '#dd6666')]:
    axes[2].hist(confidence[mask], bins=np.linspace(0.5, 1, 21), alpha=0.65, label=title, color=color)
axes[2].set(xlabel='Оценка выбранного класса', ylabel='Число фотографий', title='Уверенность не гарантирует правильность')
axes[2].legend()
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'prediction_analysis.png', bbox_inches='tight')
plt.show()


## 11. Выводы
### 11.1. Результаты эксперимента

Выводы ниже формируются из результатов текущего запуска: объём очищенных данных, выбранная эпоха,
качество на независимом test и направления ошибок. Сравнение train/validation относится к диагностике обучения;
тестовые показатели относятся к уже выбранной модели. Показатели предыдущего запуска сюда не подставляются.


In [ ]:
train_acc_best = history_dict['accuracy'][BEST_EPOCH - 1]
val_acc_best = history_dict['val_accuracy'][BEST_EPOCH - 1]
val_loss_best = history_dict['val_loss'][BEST_EPOCH - 1]
gap = train_acc_best - val_acc_best
error_count = int((y_true != y_pred).sum())
if gap > 0.05:
    learning_note = ('На выбранной эпохе train accuracy выше validation более чем на 5 процентных пунктов. '
                     'Это сигнал возможного переобучения, который следует оценивать вместе с кривыми loss.')
else:
    learning_note = ('На выбранной эпохе большого превосходства train accuracy над validation нет. '
                     'Это не доказывает отсутствие переобучения: важна динамика validation loss.')
if history_dict['val_loss'][-1] > val_loss_best + 0.01:
    learning_note += ' Последняя эпоха имеет более высокий validation loss, поэтому используется более ранний лучший checkpoint.'
lines = [
    '### Что удалось получить',
    f'Из {len(raw_df):,} фотографий осталось {len(clean_df):,} уникальных пригодных изображений. '
    f'Исключено повреждённых: {len(broken_df)}, повторных: {len(duplicate_df)}, с конфликтующими метками: {len(conflict_df)}.',
    f'Train: {len(train_df):,}; validation: {len(val_df):,}; test: {len(test_df):,}. Хеши между наборами не пересекаются.',
    f'Собственная CNN обучена с нуля за {len(history_dict["loss"])} эпох. '
    f'Лучшая эпоха — {BEST_EPOCH}, validation loss — {val_loss_best:.4f}, validation accuracy — {val_acc_best:.2%}.',
    '### Качество на новых данных',
    f'Test accuracy — **{test_accuracy:.2%}**: правильно распознано {len(y_true) - error_count} из {len(y_true)} фотографий. '
    f'Ошибок — {error_count}; test loss — {test_result["loss"]:.4f}.',
]
for label, name in enumerate(CLASS_NAMES):
    lines.append(f'**{name}:** precision — {precision[label]:.2%}, recall — {recall[label]:.2%}, F1 — {f1[label]:.2%}; '
                 f'в тесте {int(support[label])} изображений.')
lines.extend([
    f'Кошки ошибочно названы собаками {int(matrix[0, 1])} раз; собаки названы кошками {int(matrix[1, 0])} раз. '
    'Матрица и примеры ошибок показывают, какие случаи требуют дальнейшего анализа.',
    '### Что говорят кривые обучения', learning_note,
    'Train измеряется при аугментациях и Dropout, а validation — без них. Поэтому validation может быть выше train даже при корректном обучении.',
    '### Достигнутая цель',
    'Реализован полный цикл: проверка данных, разделение без точных дубликатов, обучение собственной CNN, независимая оценка и распознавание загруженной фотографии. '
    'Метрики характеризуют только этот датасет и это разделение, а не любые фотографии из интернета.',
])
conclusion = (chr(10) * 2).join(lines)
display(Markdown(conclusion))
(ARTIFACT_DIR / 'conclusions.md').write_text(conclusion, encoding='utf-8')

# Сохраняем графики, метрики и выводы вместе с моделью для защиты проекта.
shutil.copytree(ARTIFACT_DIR, SAVED_RUN_DIR / 'results', dirs_exist_ok=True)
print('Графики, метрики и выводы сохранены в Google Drive:', SAVED_RUN_DIR / 'results')


### 11.2. Ограничения и дальнейшая работа

Классификатор обучен на двух классах и всегда выбирает один из них: посторонний объект также получит оценки кошки и собаки.
Проценты sigmoid не проверялись на калибровку. Сеть может ошибаться уверенно, особенно при необычном фоне, ракурсе или частично видимом животном.

Удаление точных копий защищает от прямой утечки данных, но не распознаёт похожие кадры, другие JPEG-сжатия и серии одного животного.
Случайное разделение может поэтому давать более оптимистичное качество, чем проверка на независимом источнике.
Метки из имён могут содержать ошибки; уменьшение до 128×128 и растягивание теряют детали и изменяют пропорции.

Улучшения без предобученных весов: поиск похожих изображений и разделение по сериям; ручная проверка меток;
сохранение пропорций с padding; расширение данных; подбор числа фильтров, Dropout и скорости обучения по validation.
Для проверки улучшенной версии нужен новый независимый тест: текущий test уже изучен.

Практический результат проекта — понятный и воспроизводимый учебный эксперимент. Для ответственных решений его качества и проверки недостаточно.
